In [0]:
dbutils.widgets.text("catalog_name", "dbw_chinook_team",  "Catalog")
dbutils.widgets.text("schema_name",  "chinook_bronze",     "Schema")
dbutils.widgets.text("base_path",    "/Volumes/dbw_chinook_team/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name",   "all",                "Table Name")

In [0]:
from datetime import datetime

CATALOG     = dbutils.widgets.get("catalog_name")
SCHEMA      = dbutils.widgets.get("schema_name")
TABLE_PARAM = dbutils.widgets.get("table_name")
BRONZE      = f"{CATALOG}.{SCHEMA}"

print(f"✅ Bronze: {BRONZE}")

In [0]:
# Read latest file locations from execution log
log_df = spark.sql(f"""
    SELECT table_name, file_location
    FROM {BRONZE}.pipeline_execution_log
    WHERE status = 'SUCCESS'
    AND execution_time = (
        SELECT MAX(execution_time) 
        FROM {BRONZE}.pipeline_execution_log l2
        WHERE l2.table_name = pipeline_execution_log.table_name
        AND l2.status = 'SUCCESS'
    )
""")

if TABLE_PARAM != "all":
    log_df = log_df.filter(f"table_name = '{TABLE_PARAM}'")

files = [(row.table_name, row.file_location) 
         for row in log_df.collect()]

print(f"✅ Files to load: {len(files)}")
for t, f in files:
    print(f"   {t} → {f}")

In [0]:
print("\n=== RAW → BRONZE LOAD ===")

for table_name, file_location in files:
    print(f"\nLoading: {table_name}")
    try:
        # Read from Raw Volume
        raw_df = spark.read.parquet(file_location)
        
        target = f"{BRONZE}.{table_name.lower()}"
        
        # Write Delta — overwrite (today's snapshot)
        raw_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target)
        
        count = spark.table(target).count()
        print(f"  ✅ {target}: {count} rows")
        
    except Exception as e:
        print(f"  ❌ {table_name} FAILED: {e}")

print("\n=== BRONZE LOAD COMPLETE ===")

In [0]:
print("\n=== BRONZE VALIDATION ===")
for table_name, _ in files:
    count = spark.table(
        f"{BRONZE}.{table_name.lower()}"
    ).count()
    print(f"  ✅ {table_name:<20} {count:>6} rows")